# 2. Searching and downloading

How to narrow a search to exactly the scenes you want, and get the right product
for each one onto disk.

In [ ]:
# Point at the hosted API, or a local build for testing.
#   BASE = 'http://localhost:8000/api/v1'   # after: python3 -m http.server 8000
BASE = None

from opensartriad import Catalog

cat = Catalog(BASE) if BASE else Catalog()
cat

## Narrowing a search

Start broad, then add filters. Each is optional.

In [ ]:
aoi = (72.8, 18.9, 73.1, 19.3)   # Mumbai

broad = cat.search(bbox=aoi)
print('all time           :', len(broad))

recent = cat.search(bbox=aoi, start='2024-01-01')
print('2024 onwards       :', len(recent))

spot = cat.search(bbox=aoi, start='2024-01-01', mode='spotlight')
print('spotlight only     :', len(spot))

asc = cat.search(bbox=aoi, start='2024-01-01', mode='spotlight', orbit='ascending')
print('ascending passes   :', len(asc))

### Acquisition geometry

`orbit` and `look` matter when you need a consistent viewing geometry across dates.
Filtering by them is how you build a comparable stack.

In [ ]:
for orbit in ('ascending', 'descending'):
    for look in ('left', 'right'):
        n = len(cat.search(bbox=aoi, orbit=orbit, look=look))
        print(f'{orbit:11} / {look:5} : {n}')

## Product families

Providers name the same kind of product differently. Asking for `SLC` by name
returns **nothing** from Umbra, even though every Umbra scene carries complex data,
because Umbra labels it `SICD`.

| Family | Resolves in order |
|---|---|
| `detected` | `GEO` -> `GRD` -> `GEC` -> `SIDD` |
| `complex` | `SLC` -> `SICD` |
| `phase` | `CPHD` |
| `visual` | `CSI` -> `VID` |

Ask for what the product *is*, and each scene resolves to its provider's equivalent.

In [ ]:
# One scene per provider, to compare how a family resolves
for provider in ('iceye', 'umbra', 'capella'):
    hit = cat.search(providers=provider, limit=1)
    if not hit:
        continue
    s = hit[0]
    print(f'{provider:8} formats={s.formats}')
    for fam in ('detected', 'complex', 'phase', 'visual'):
        print(f'   {fam:9} -> {s.resolve(fam)}')
    print()

## Always dry run first

SAR products are **large**: a single GRD can exceed 1 GB. Check the file count
before committing to a download.

In [ ]:
picks = cat.search(bbox=aoi, start='2024-01-01', limit=3)
print(picks)

jobs = picks.download_urls(family='complex')
print(f'\n{len(jobs)} files would be fetched:')
for j in jobs:
    print(f"  [{j['kind']:8}] {j['provider']}/{j['url'].rsplit('/', 1)[-1]}")

In [ ]:
# The same thing through download(), which prints a summary and fetches nothing
picks.download('data/', family='complex', dry_run=True)

## Downloading

Files land in `data/<provider>/`, and each provider's **metadata sidecar is saved
next to every data file**, so the archive documents itself.

Downloads are resumable: existing files are skipped, partial writes go to `.part`
and are only renamed on success, and one failed asset does not abort the batch.

> Uncomment to actually download. Expect gigabytes.

In [ ]:
# paths = picks.download('data/', family='complex')
# for p in paths:
#     print(p, p.stat().st_size / 1e6, 'MB')

### Exact formats instead of families

When you need one specific product and no fallback.

In [ ]:
picks.download('data/', formats=['SICD'], dry_run=True)

## Look before you download

Plotting the selection is a quick sanity check that you picked the right scenes
before spending bandwidth on them. Needs `pip install 'open-sar-triad[plot]'`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

picks.plot_coverage(bbox=(65, 12, 80, 26), footprints=True)
plt.show()

In [ ]:
# A single scene, with enough surrounding geography to place it
picks[0].plot_footprint(pad=8)
plt.show()

## Saving the search itself

Useful for sharing an AOI selection or feeding another tool.

In [ ]:
picks.save_geojson('selection.geojson')

import json
gj = json.load(open('selection.geojson'))
print(gj['type'], len(gj['features']), 'features')

---

Next: **03_stac_and_analysis.ipynb** for STAC interop and analysing coverage.